# Capítulo 7 — Teoria dos jogos: a disputa com o concorrente

## Missão
A Boreal e a rival disputam participação de mercado. Cada uma escolhe uma estratégia (preço, propaganda, qualidade...) sem saber a da outra. O ganho de uma é a perda da outra: um **jogo de soma zero**. Você vai:
- procurar o **ponto de sela** (maximin = minimax);
- eliminar estratégias **dominadas**;
- resolver os jogos sem ponto de sela com **estratégias mistas**, pelo método gráfico e por programação linear.

**No fim deste caderno você terá:** o módulo `gabarito_otimizador/jogos.py` (ponto de sela, dominância e estratégia mista ótima via programação linear) e o "mix" de estratégias da Boreal.

**Leitura essencial (opcional, livro em inglês):**
- 7.2 Two-Person, Zero-Sum Games (with Saddle Point) · PDF p. 238
- 7.3 The Maximin-Minimax Principle · PDF p. 239
- 7.4 Two-Person, Zero-Sum Games (without Saddle Point) · PDF p. 243
- 7.5 The Principle of Dominance · PDF p. 248
- 7.6 Graphical Method for Solving Games · PDF p. 253

**Gabarito:** a mesma trilha com o código pronto. Tente primeiro em `praticar/`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote gabarito_otimizador/
os.makedirs("../gabarito_otimizador", exist_ok=True)
open("../gabarito_otimizador/__init__.py", "a").close()        # gabarito_otimizador/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: gabarito_otimizador")

## Ideia em 30 segundos
O jogador das linhas (a Boreal) pensa "qual o pior que me acontece em cada linha?" e escolhe a melhor delas: o **maximin**. O das colunas faz o contrário: o **minimax**. Se os dois coincidem, há **ponto de sela** e a estratégia pura resolve.

Se não coincidem, cada um deve **sortear** as estratégias com certas probabilidades. Achar essas probabilidades é um problema de programação linear (von Neumann).

### 7.1 **O módulo de jogos.** Crie `jogos.py` com:
- `ponto_de_sela(A)`: devolve `(maximin, minimax, posicao ou None)`;
- `remover_dominadas(A, linhas, colunas)`: repete até não mudar. Remove a linha que é ≤ outra em tudo e a coluna que é ≥ outra em tudo, e devolve a matriz reduzida com os nomes;
- `mista(A)`: some uma constante para deixar tudo positivo, resolva `min Σx` com `Aᵀx ≥ 1` e `x ≥ 0` (`linprog`) e devolva as probabilidades da linha e o **valor do jogo**. Faça o mesmo para as colunas.

_A célula grava `gabarito_otimizador/jogos.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../gabarito_otimizador/jogos.py
"""Jogos de soma zero com dois jogadores."""
import numpy as np
from scipy.optimize import linprog


def ponto_de_sela(A):
    A = np.asarray(A, float)
    maximin, minimax = A.min(axis=1).max(), A.max(axis=0).min()
    if maximin == minimax:
        i = int(A.min(axis=1).argmax())
        j = int(A.max(axis=0).argmin())
        return float(maximin), float(minimax), (i, j)
    return float(maximin), float(minimax), None


def remover_dominadas(A, linhas, colunas):
    A, linhas, colunas = np.asarray(A, float), list(linhas), list(colunas)
    mudou = True
    while mudou:
        mudou = False
        for i in range(len(A)):
            if any(k != i and np.all(A[i] <= A[k]) for k in range(len(A))):
                A, linhas, mudou = np.delete(A, i, 0), linhas[:i] + linhas[i + 1:], True
                break
        if mudou:
            continue
        for j in range(A.shape[1]):
            if any(k != j and np.all(A[:, j] >= A[:, k]) for k in range(A.shape[1])):
                A, colunas, mudou = np.delete(A, j, 1), colunas[:j] + colunas[j + 1:], True
                break
    return A, linhas, colunas


def mista(A):
    A = np.asarray(A, float)
    k = 1 - A.min()
    B = A + k
    m, n = B.shape
    lin = linprog(np.ones(m), A_ub=-B.T, b_ub=-np.ones(n), method="highs")
    col = linprog(-np.ones(n), A_ub=B, b_ub=np.ones(m), method="highs")
    v = 1 / lin.x.sum()
    return lin.x * v, col.x * (1 / col.x.sum()), v - k

### 7.2 **Ponto de sela?** Mostre o `JOGO` com os nomes das estratégias, o mínimo de cada linha, o máximo de cada coluna, o maximin e o minimax. Existe ponto de sela?

In [ ]:
from gabarito_otimizador.jogos import ponto_de_sela, remover_dominadas, mista
tab = pd.DataFrame(JOGO, index=ESTRATEGIAS_BOREAL, columns=ESTRATEGIAS_RIVAL)
display(tab.assign(min_linha=tab.min(axis=1)))
print("máximo das colunas:", tab.max().to_dict())
print(ponto_de_sela(JOGO))

### 7.3 **Um jogo com sela.** Teste também a matriz `[[4, 2, 5], [3, 1, 2], [6, 3, 4]]`: o ponto de sela diz a estratégia pura de cada um e o valor do jogo.

In [ ]:
print(ponto_de_sela([[4, 2, 5], [3, 1, 2], [6, 3, 4]]))

### 7.4 **Dominância.** Reduza o `JOGO` removendo as estratégias dominadas. Que estratégias da Boreal e da rival **nunca** devem ser usadas?

In [ ]:
Ar, lin, col = remover_dominadas(JOGO, ESTRATEGIAS_BOREAL, ESTRATEGIAS_RIVAL)
display(pd.DataFrame(Ar, index=lin, columns=col))
print("Boreal descarta:", set(ESTRATEGIAS_BOREAL) - set(lin), "| rival descarta:", set(ESTRATEGIAS_RIVAL) - set(col))

### 7.5 **Estratégia mista.** Resolva o `JOGO` completo com `mista`. Mostre com que probabilidade a Boreal deve usar cada estratégia, o mesmo para a rival e o valor do jogo. Confira que o resultado é o mesmo usando a matriz reduzida.

In [ ]:
px, py, v = mista(JOGO)
print("Boreal:", dict(zip(ESTRATEGIAS_BOREAL, px.round(3).tolist()))); print("rival:", dict(zip(ESTRATEGIAS_RIVAL, py.round(3).tolist())))
print("valor do jogo:", round(v, 3))
print("reduzida, valor:", round(mista(Ar)[2], 3))

### 7.6 **Método gráfico (2 × n).** Na matriz reduzida, se a Boreal ficou com 2 estratégias, desenhe o ganho esperado contra cada coluna da rival em função de `p` (a probabilidade da 1ª linha). O ótimo é o **ponto mais alto da envoltória inferior**. Compare com o `mista`.

In [ ]:
import matplotlib.pyplot as plt
if Ar.shape[0] == 2:
    ps = np.linspace(0, 1, 201)
    linhas_j = [p_ * Ar[0, j] + (1 - p_) * Ar[1, j] for j in range(Ar.shape[1]) for p_ in [ps]]
    for j, y in enumerate(linhas_j):
        plt.plot(ps, y, label=f"rival: {col[j]}")
    env = np.min(linhas_j, axis=0); k = int(env.argmax())
    plt.plot(ps, env, "k", lw=3); plt.plot(ps[k], env[k], "r*", ms=15); plt.xlabel(f"p = P({lin[0]})"); plt.legend(); plt.show()
    print("gráfico: p =", round(ps[k], 3), "| valor", round(env[k], 3), "| mista:", mista(Ar)[0].round(3))
else:
    print("a matriz reduzida tem", Ar.shape[0], "linhas; o método gráfico vale para 2 × n")

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
